# Parte 1: Prepacion del entorno de trabajo y de los datos - *no ejecutar*

## Comprobación del entorno
Comprobamos que el notebook se está ejecutando con la versión correcta de Python y con el entorno virtual del proyecto. Esto evita errores de dependencias y asegura que todo el trabajo se realiza sobre el mismo entorno reproducible.

In [1]:
import sys
print(sys.version)
print(sys.executable)

3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
c:\proyectosGithub\PrediccionRetrasosVuelos-ML\venv\Scripts\python.exe


## Procesamiento de los CSV grandes (data raw) por fragmentos
Dado que los CSV originales son demasiado grandes para cargarlos de una vez, se procesan por bloques de 300.000 filas. En cada bloque se eliminan vuelos cancelados y desviados, se eliminan registros sin retraso de llegada y se crea la variable objetivo binaria `delay_15`, que vale 1 si el retraso es igual o superior a 15 minutos y 0 en caso contrario.

Además, cada bloque limpio se guarda en formato parquet, que ocupa menos espacio y permite una lectura posterior más rápida que los CSV originales.

Este bloque solo se ejecuta si se quiere reproducir el preprocesado completo a partir de los archivos en `data/raw`. Para continuar con el proyecto no es necesario ejecutarlo si ya existe la carpeta `data/processed`.

Antes de procesar los datos, se revisan las columnas reales de uno de los CSV para evitar errores de nombres y confirmar qué variables están disponibles en el dataset. Esta comprobación es necesaria porque el preprocesado posterior depende de usar exactamente esas columnas.

In [1]:
from pathlib import Path
import pandas as pd

YEAR = 2023  # cambiar a 2024 si se quiere revisar ese año
DATA_DIR = Path("data/raw") / str(YEAR)

file = next(DATA_DIR.glob("*.csv"))

df0 = pd.read_csv(file, nrows=5)
df0.columns = df0.columns.str.strip()

print(f"Columnas reales para {file.name}:")
for c in df0.columns:
    print(repr(c))

Columnas reales para T_ONTIME_REPORTING_APR.csv:
'YEAR'
'MONTH'
'DAY_OF_MONTH'
'DAY_OF_WEEK'
'OP_UNIQUE_CARRIER'
'ORIGIN'
'ORIGIN_STATE_ABR'
'DEST'
'DEST_STATE_ABR'
'CRS_DEP_TIME'
'DEP_TIME_BLK'
'ARR_DELAY'
'CANCELLED'
'DIVERTED'
'DISTANCE'
'DISTANCE_GROUP'


### Definición de rutas, variables y tipos
En este bloque se fijan las rutas de entrada y salida, se seleccionan las columnas que se van a conservar y se asignan tipos de datos más ligeros. El objetivo es reducir el consumo de memoria y dejar preparado un esquema consistente para procesar todos los ficheros.

In [2]:
from pathlib import Path
import pandas as pd

# Columnas reales del dataset
USECOLS = [
    "YEAR",
    "MONTH",
    "DAY_OF_MONTH",
    "DAY_OF_WEEK",
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "ORIGIN_STATE_ABR",
    "DEST",
    "DEST_STATE_ABR",
    "CRS_DEP_TIME",
    "DEP_TIME_BLK",
    "ARR_DELAY",
    "CANCELLED",
    "DIVERTED",
    "DISTANCE",
    "DISTANCE_GROUP",
]

# Tipos para reducir memoria
DTYPES = {
    "YEAR": "int16",
    "MONTH": "int8",
    "DAY_OF_MONTH": "int8",
    "DAY_OF_WEEK": "int8",
    "OP_UNIQUE_CARRIER": "category",
    "ORIGIN": "category",
    "ORIGIN_STATE_ABR": "category",
    "DEST": "category",
    "DEST_STATE_ABR": "category",
    "CRS_DEP_TIME": "Int32",   # mejor nullable por si hay valores raros o nulos
    "DEP_TIME_BLK": "category",
    "ARR_DELAY": "float32",
    "CANCELLED": "int8",
    "DIVERTED": "int8",
    "DISTANCE": "float32",
    "DISTANCE_GROUP": "int8",
}

In [3]:

# Tamaño de chunk: buen equilibrio
CHUNK_SIZE = 300_000

def process_year(year: int) -> None:
    data_dir = Path("data/raw") / str(year)
    output_dir = Path("data/processed") / str(year)
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_files = sorted(data_dir.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No se encontraron CSV en {data_dir.resolve()}")

    print(f"\n========== PROCESANDO AÑO {year} ==========")

    for file in csv_files:
        print(f"\nProcesando {file.name}...")

        # Validación de cabecera
        header = pd.read_csv(file, nrows=0)
        header.columns = header.columns.str.strip()
        missing = [col for col in USECOLS if col not in header.columns]
        if missing:
            print(f"  Saltado: faltan columnas en {file.name}: {missing}")
            continue

        total_rows_in = 0
        total_rows_out = 0

        for i, chunk in enumerate(
            pd.read_csv(
                file,
                usecols=USECOLS,
                dtype=DTYPES,
                chunksize=CHUNK_SIZE,
                low_memory=False,
            )
        ):
            chunk.columns = chunk.columns.str.strip()

            rows_before = len(chunk)
            total_rows_in += rows_before

            # Limpieza básica
            chunk = chunk[(chunk["CANCELLED"] == 0) & (chunk["DIVERTED"] == 0)]
            chunk = chunk[chunk["ARR_DELAY"].notna()].copy()

            # Target binario
            chunk["delay_15"] = (chunk["ARR_DELAY"] >= 15).astype("int8")

            rows_after = len(chunk)
            total_rows_out += rows_after

            # Guardado con año en el nombre
            out_path = output_dir / f"{file.stem}_{year}_chunk_{i:03d}.parquet"
            chunk.to_parquet(out_path, index=False)

            print(
                f"  Chunk {i:03d}: "
                f"{rows_before:,} filas -> {rows_after:,} filas guardadas "
                f"en {out_path.name}"
            )

        print(
            f"Terminado {file.name}: "
            f"{total_rows_in:,} filas leídas, {total_rows_out:,} filas guardadas."
        )

    print(f"\nProceso completo para {year}.")

In [4]:
process_year(2023)
process_year(2024)


========== PROCESANDO AÑO 2023 ==========

Procesando T_ONTIME_REPORTING_APR.csv...
  Chunk 000: 300,000 filas -> 293,579 filas guardadas en T_ONTIME_REPORTING_APR_2023_chunk_000.parquet
  Chunk 001: 261,441 filas -> 256,670 filas guardadas en T_ONTIME_REPORTING_APR_2023_chunk_001.parquet
Terminado T_ONTIME_REPORTING_APR.csv: 561,441 filas leídas, 550,249 filas guardadas.

Procesando T_ONTIME_REPORTING_AUG.csv...
  Chunk 000: 300,000 filas -> 294,321 filas guardadas en T_ONTIME_REPORTING_AUG_2023_chunk_000.parquet
  Chunk 001: 300,000 filas -> 294,873 filas guardadas en T_ONTIME_REPORTING_AUG_2023_chunk_001.parquet
  Chunk 002: 2,987 filas -> 2,948 filas guardadas en T_ONTIME_REPORTING_AUG_2023_chunk_002.parquet
Terminado T_ONTIME_REPORTING_AUG.csv: 602,987 filas leídas, 592,142 filas guardadas.

Procesando T_ONTIME_REPORTING_DEC.csv...
  Chunk 000: 300,000 filas -> 298,893 filas guardadas en T_ONTIME_REPORTING_DEC_2023_chunk_000.parquet
  Chunk 001: 270,394 filas -> 268,048 filas gua

# Parte 2: Inicio de trabajo desde datos procesados

A partir de este punto, el notebook carga directamente los archivos parquet de `data/processed`. 

## Construcción de subconjuntos de trabajo
Se generan varios subconjuntos de datos con funciones diferenciadas para mantener una organización metodológica clara y un coste computacional razonable. 

Para el **análisis exploratorio** se construyen ``eda_2023`` y ``eda_2024`` mediante un muestreo estratificado del **10%** de cada año respecto a la variable objetivo ``delay_15``. Este porcentaje se considera suficiente para conservar la estructura general del problema, las proporciones entre clases y los patrones principales por mes, aerolínea o aeropuerto, pero reduciendo de forma importante el volumen de datos sobre el que se calculan tablas y visualizaciones. 

Además, se crea ``train_sample_2023`` como una submuestra estratificada del **20%** del conjunto de entrenamiento de 2023, pensada para experimentación rápida y para modelos más costosos en tiempo y memoria. 
Por otra parte, el conjunto de 2023 se divide temporalmente en ``train_2023`` y ``valid_2023``, usando los meses enero-septiembre para **entrenamiento** y octubre-diciembre para **validación interna**, mientras que ``test_2024`` se reserva como conjunto de **evaluación final**. 

Esta partición temporal permite entrenar con datos pasados y evaluar con datos posteriores, reproduciendo de forma más realista un escenario de predicción.

In [10]:
from pathlib import Path
import pandas as pd

SAMPLES_DIR = Path("data/samples")
FINAL_DIR = Path("data/final")

def load_processed_year(year: int) -> pd.DataFrame:
    processed_dir = Path("data/processed") / str(year)
    parquet_files = sorted(processed_dir.glob("*.parquet"))

    if not parquet_files:
        raise FileNotFoundError(f"No se encontraron parquet en {processed_dir.resolve()}")

    df = pd.concat((pd.read_parquet(f) for f in parquet_files), ignore_index=True)
    return df

def stratified_sample(df: pd.DataFrame, target: str, frac: float, random_state: int = 42) -> pd.DataFrame:
    sampled = (
        df.groupby(target, group_keys=False)
          .sample(frac=frac, random_state=random_state)
          .reset_index(drop=True)
          .copy()
    )
    return sampled

# Cargar años
df_2023 = load_processed_year(2023)
df_2024 = load_processed_year(2024)

print("2023:", df_2023.shape)
print("2024:", df_2024.shape)

# Samples para EDA
eda_2023 = stratified_sample(df_2023, target="delay_15", frac=0.10, random_state=42)
eda_2024 = stratified_sample(df_2024, target="delay_15", frac=0.10, random_state=42)

eda_2023.to_parquet(SAMPLES_DIR / "eda_2023.parquet", index=False)
eda_2024.to_parquet(SAMPLES_DIR / "eda_2024.parquet", index=False)

# Split temporal dentro de 2023
train_2023 = df_2023[df_2023["MONTH"].between(1, 9)].copy()
valid_2023 = df_2023[df_2023["MONTH"].between(10, 12)].copy()
test_2024 = df_2024.copy()

train_2023.to_parquet(FINAL_DIR / "train_2023.parquet", index=False)
valid_2023.to_parquet(FINAL_DIR / "valid_2023.parquet", index=False)
test_2024.to_parquet(FINAL_DIR / "test_2024.parquet", index=False)

# Muestra del train para experimentos rápidos o modelos caros
train_sample_2023 = stratified_sample(train_2023, target="delay_15", frac=0.20, random_state=42)
train_sample_2023.to_parquet(SAMPLES_DIR / "train_sample_2023.parquet", index=False)


2023: (6743403, 17)
2024: (6965267, 17)


In [11]:
print("\nArchivos creados:")
print(" - data/samples/eda_2023.parquet", eda_2023.shape)
print(" - data/samples/eda_2024.parquet", eda_2024.shape)
print(" - data/final/train_2023.parquet", train_2023.shape)
print(" - data/final/valid_2023.parquet", valid_2023.shape)
print(" - data/final/test_2024.parquet", test_2024.shape)
print(" - data/samples/train_sample_2023.parquet", train_sample_2023.shape)


Archivos creados:
 - data/samples/eda_2023.parquet (674340, 17)
 - data/samples/eda_2024.parquet (696527, 17)
 - data/final/train_2023.parquet (5018046, 17)
 - data/final/valid_2023.parquet (1725357, 17)
 - data/final/test_2024.parquet (6965267, 17)
 - data/samples/train_sample_2023.parquet (1003610, 17)


### Comprobación de calidad de los subconjuntos y muestras
Cargamos los datasets completos y las muestras para comprobar si el muestreo mantiene una estructura similar en la variable objetivo, en la distribución temporal y en las variables principales.

In [12]:
# =========================
# FUNCIONES AUXILIARES
# =========================
def add_dep_hour(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["DEP_HOUR"] = (df["CRS_DEP_TIME"] // 100).astype("Int16")
    return df


def compare_target_distribution(full_df: pd.DataFrame, sample_df: pd.DataFrame, target: str = "delay_15") -> pd.DataFrame:
    full_dist = full_df[target].value_counts(normalize=True).sort_index().rename("full_pct")
    sample_dist = sample_df[target].value_counts(normalize=True).sort_index().rename("sample_pct")
    out = pd.concat([full_dist, sample_dist], axis=1).fillna(0)
    out["abs_diff_pp"] = (out["sample_pct"] - out["full_pct"]).abs() * 100
    return out


def compare_categorical_distribution(
    full_df: pd.DataFrame,
    sample_df: pd.DataFrame,
    col: str,
    top_n: int | None = None
) -> pd.DataFrame:
    full_dist = full_df[col].value_counts(normalize=True, dropna=False)
    sample_dist = sample_df[col].value_counts(normalize=True, dropna=False)

    if top_n is not None:
        top_categories = full_dist.head(top_n).index
        full_dist = full_dist.loc[top_categories]
        sample_dist = sample_dist.reindex(top_categories, fill_value=0)

    out = pd.concat(
        [full_dist.rename("full_pct"), sample_dist.rename("sample_pct")],
        axis=1
    ).fillna(0)

    out["abs_diff_pp"] = (out["sample_pct"] - out["full_pct"]).abs() * 100
    out = out.sort_values("full_pct", ascending=False)
    return out


def compare_numeric_summary(full_df: pd.DataFrame, sample_df: pd.DataFrame, col: str) -> pd.DataFrame:
    percentiles = [0.05, 0.25, 0.5, 0.75, 0.95]

    def summarize(s: pd.Series) -> pd.Series:
        s = pd.to_numeric(s, errors="coerce").dropna()
        return pd.Series({
            "count": s.shape[0],
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "p05": s.quantile(percentiles[0]),
            "p25": s.quantile(percentiles[1]),
            "p50": s.quantile(percentiles[2]),
            "p75": s.quantile(percentiles[3]),
            "p95": s.quantile(percentiles[4]),
            "max": s.max(),
        })

    full_stats = summarize(full_df[col]).rename("full")
    sample_stats = summarize(sample_df[col]).rename("sample")

    out = pd.concat([full_stats, sample_stats], axis=1)
    out["abs_diff"] = (out["sample"] - out["full"]).abs()
    return out


def quality_report(
    full_df: pd.DataFrame,
    sample_df: pd.DataFrame,
    name_full: str,
    name_sample: str
) -> None:
    print("=" * 80)
    print(f"COMPARACIÓN: {name_sample} vs {name_full}")
    print("=" * 80)
    print(f"Tamaño completo: {full_df.shape}")
    print(f"Tamaño muestra : {sample_df.shape}")
    print(f"Fracción real  : {len(sample_df) / len(full_df):.2%}")
    print()

    print("1) Distribución de la variable objetivo: delay_15")
    target_table = compare_target_distribution(full_df, sample_df, target="delay_15")
    print(target_table)
    print(f"Máx. diferencia absoluta en delay_15: {target_table['abs_diff_pp'].max():.3f} pp")
    print()

    cat_cols = [
        ("MONTH", None),
        ("DAY_OF_WEEK", None),
        ("DEP_TIME_BLK", None),
        ("DEP_HOUR", None),
        ("OP_UNIQUE_CARRIER", None),
        ("DISTANCE_GROUP", None),
        ("ORIGIN", 10),
        ("DEST", 10),
    ]

    print("2) Distribuciones categóricas principales")
    for col, top_n in cat_cols:
        if col in full_df.columns and col in sample_df.columns:
            print(f"\n--- {col} ---")
            table = compare_categorical_distribution(full_df, sample_df, col=col, top_n=top_n)
            print(table.head(15))
            print(f"Máx. diferencia absoluta en {col}: {table['abs_diff_pp'].max():.3f} pp")

    print()
    num_cols = ["DISTANCE", "ARR_DELAY"]

    print("3) Resumen de variables numéricas")
    for col in num_cols:
        if col in full_df.columns and col in sample_df.columns:
            print(f"\n--- {col} ---")
            print(compare_numeric_summary(full_df, sample_df, col=col).round(4))

    print("\n")

In [13]:
df_2023 = add_dep_hour(df_2023)
df_2024 = add_dep_hour(df_2024)
eda_2023 = add_dep_hour(eda_2023)
eda_2024 = add_dep_hour(eda_2024)
train_2023 = add_dep_hour(train_2023)
train_sample_2023 = add_dep_hour(train_sample_2023)

In [15]:
quality_report(df_2023, eda_2023, "2023 completo", "eda_2023")
quality_report(df_2024, eda_2024, "2024 completo", "eda_2024")
quality_report(train_2023, train_sample_2023, "train_2023", "train_sample_2023")

COMPARACIÓN: eda_2023 vs 2023 completo
Tamaño completo: (6743403, 18)
Tamaño muestra : (674340, 18)
Fracción real  : 10.00%

1) Distribución de la variable objetivo: delay_15
          full_pct  sample_pct  abs_diff_pp
delay_15                                   
0         0.794362    0.794362     0.000024
1         0.205638    0.205638     0.000024
Máx. diferencia absoluta en delay_15: 0.000 pp

2) Distribuciones categóricas principales

--- MONTH ---
       full_pct  sample_pct  abs_diff_pp
MONTH                                   
10     0.088383    0.088258     0.012497
8      0.087811    0.087975     0.016435
7      0.086760    0.086399     0.036150
5      0.085332    0.085460     0.012772
3      0.084754    0.084707     0.004786
12     0.084073    0.084039     0.003422
6      0.083460    0.083900     0.043987
11     0.083402    0.083421     0.001887
9      0.083176    0.082995     0.018044
4      0.081598    0.081972     0.037388
1      0.078180    0.078091     0.008850
2      0.07

Las comprobaciones de calidad del muestreo muestran que las submuestras generadas mantienen con alta fidelidad la estructura de los conjuntos originales. En particular, la distribución de la variable objetivo delay_15 se conserva prácticamente sin variación, y las diferencias observadas en variables temporales, categóricas y operativas principales son muy pequeñas, en general inferiores a una décima de punto porcentual. En las variables numéricas, medias y percentiles se mantienen estables, aunque los valores máximos de ARR_DELAY difieren más debido a la menor presencia de retrasos extremadamente raros en las muestras. En conjunto, las muestras pueden considerarse suficientemente representativas para EDA y experimentación inicial.

Con esta primera fase se ha dejado preparado un conjunto de datos limpio, comprimido y manejable para continuar con el análisis exploratorio. El objetivo de este preprocesado no es todavía modelizar, sino asegurar que la base de trabajo sea consistente, eficiente y representativa.